# Next word recommender

In [1]:
# loading the required libraries
import pandas as pd
import numpy as np
import re
import pickle
import random
from tqdm import tqdm

In [2]:
data = pd.read_csv("sample_reuters_dataset.csv")

In [3]:
data

,sentence_number,sentence_text
0,0,ASIAN EXPORTERS FEAR DAMAGE FROM U . S .- JAPA...
1,1,They told Reuter correspondents in Asian capit...
2,2,But some exporters said that while the conflic...
3,3,The U . S . Has said it will impose 300 mln dl...
4,4,Unofficial Japanese estimates put the impact o...
...,...,...
9995,9995,"In addition , British Printing and Communicati..."
9996,9996,Salomon said in a filing with the Securities a...
9997,9997,If the court decides they should be converted ...
9998,9998,Harcourt is asking the court to rule the compa...


In [5]:
# text cleaning
dialogs = data["sentence_text"].tolist()
dialogs_clean = []

for i in dialogs:
  # remove everything except alphabets, ' and white spaces
  i = re.sub("[^a-zA-Z' ]", "", i)
  # convert text to lowercase
  i = i.lower()
  # add cleaned text to the list
  dialogs_clean.append(i)

In [6]:
data["clean_text"] = dialogs_clean

# Creating N-gram, Bi-gram , Tri-gram

In [8]:
# function to create unigrams
# taking a sentence as input
def create_unigram(sentence):
    # creating tokens from the sentence
    tokens = sentence.split()
    # empty list to store the unigrams
    unigram_list = []
    # number of unigrams is equal to the number of tokens in the sentence
    for i in range(len(tokens)):
        # appending each unigram in the list
        unigram_list.append(tokens[i:i+1])
    # returning the unigram list for a sentence    
    return unigram_list

# function to create bigrams
def create_bigram(sentence):
    tokens = sentence.split()
    bigram_list = []
    # number of bigrams is one less than the number of tokens in the sentence
    for i in range(len(tokens)-1):
        bigram_list.append(tokens[i:i+2])
    return bigram_list

# function to create trigrams
def create_trigram(sentence):
    tokens = sentence.split()
    trigram_list = []
    # number of trigrams is two less than the number of tokens in the sentence
    for i in range(len(tokens)-2):
        trigram_list.append(tokens[i:i+3])
    return trigram_list

In [9]:
# creating unigrams for all the sentences in the dataset 
final_unigram = []
# for each sentence
for i in range(data.shape[0]):
    # using the defined unigram function to create unigrams
    final_unigram.append(create_unigram(data['clean_text'][i]))

# adding the unigram in a seperate column in the dataset
data['unigram'] = final_unigram


# creating bigrams for all the sentences in the dataset
final_bigram = []
for i in range(data.shape[0]):
    final_bigram.append(create_bigram(data['clean_text'][i]))

data['bigram'] = final_bigram

# creating trigrams for all the sentences in the dataset
final_trigram = []
for i in range(data.shape[0]):
    final_trigram.append(create_trigram(data['clean_text'][i]))

data['trigram'] = final_trigram

In [10]:
data

,sentence_number,sentence_text,clean_text,unigram,bigram,trigram
0,0,ASIAN EXPORTERS FEAR DAMAGE FROM U . S .- JAPA...,asian exporters fear damage from u s japan r...,"[[asian], [exporters], [fear], [damage], [from...","[[asian, exporters], [exporters, fear], [fear,...","[[asian, exporters, fear], [exporters, fear, d..."
1,1,They told Reuter correspondents in Asian capit...,they told reuter correspondents in asian capit...,"[[they], [told], [reuter], [correspondents], [...","[[they, told], [told, reuter], [reuter, corres...","[[they, told, reuter], [told, reuter, correspo..."
2,2,But some exporters said that while the conflic...,but some exporters said that while the conflic...,"[[but], [some], [exporters], [said], [that], [...","[[but, some], [some, exporters], [exporters, s...","[[but, some, exporters], [some, exporters, sai..."
3,3,The U . S . Has said it will impose 300 mln dl...,the u s has said it will impose mln dlrs of...,"[[the], [u], [s], [has], [said], [it], [will],...","[[the, u], [u, s], [s, has], [has, said], [sai...","[[the, u, s], [u, s, has], [s, has, said], [ha..."
4,4,Unofficial Japanese estimates put the impact o...,unofficial japanese estimates put the impact o...,"[[unofficial], [japanese], [estimates], [put],...","[[unofficial, japanese], [japanese, estimates]...","[[unofficial, japanese, estimates], [japanese,..."
...,...,...,...,...,...,...
9995,9995,"In addition , British Printing and Communicati...",in addition british printing and communicatio...,"[[in], [addition], [british], [printing], [and...","[[in, addition], [addition, british], [british...","[[in, addition, british], [addition, british, ..."
9996,9996,Salomon said in a filing with the Securities a...,salomon said in a filing with the securities a...,"[[salomon], [said], [in], [a], [filing], [with...","[[salomon, said], [said, in], [in, a], [a, fil...","[[salomon, said, in], [said, in, a], [in, a, f..."
9997,9997,If the court decides they should be converted ...,if the court decides they should be converted ...,"[[if], [the], [court], [decides], [they], [sho...","[[if, the], [the, court], [court, decides], [d...","[[if, the, court], [the, court, decides], [cou..."
9998,9998,Harcourt is asking the court to rule the compa...,harcourt is asking the court to rule the compa...,"[[harcourt], [is], [asking], [the], [court], [...","[[harcourt, is], [is, asking], [asking, the], ...","[[harcourt, is, asking], [is, asking, the], [a..."


# Build N-gram language model

In [12]:
# for defining the N-gram model
from collections import Counter, defaultdict

# Create a placeholder for model
model = defaultdict(lambda: defaultdict(lambda: 0))

# Count frequency of co-occurance  
for i in range(data.shape[0]):
    # for each trigram pair
    for w1, w2, w3 in create_trigram(data['clean_text'][i]):
        # count the occurance of word 3, given word 1 and word 2
        model[(w1, w2)][w3] += 1

In [13]:
model

defaultdict(<function __main__.<lambda>()>,
            {('asian',
              'exporters'): defaultdict(<function __main__.<lambda>.<locals>.<lambda>()>, {'fear': 1}),
             ('exporters',
              'fear'): defaultdict(<function __main__.<lambda>.<locals>.<lambda>()>, {'damage': 1}),
             ('fear',
              'damage'): defaultdict(<function __main__.<lambda>.<locals>.<lambda>()>, {'from': 1}),
             ('damage',
              'from'): defaultdict(<function __main__.<lambda>.<locals>.<lambda>()>, {'u': 1,
                          'local': 1}),
             ('from',
              'u'): defaultdict(<function __main__.<lambda>.<locals>.<lambda>()>, {'s': 5,
                          'k': 1}),
             ('u',
              's'): defaultdict(<function __main__.<lambda>.<locals>.<lambda>()>, {'japan': 6,
                          'and': 34,
                          'move': 1,
                          'has': 7,
                          'said': 4,
          

In [17]:
dict(model['that','the'])

{'row': 1,
 'exchange': 1,
 'withdrawal': 1,
 'underlying': 2,
 'two': 3,
 'situation': 4,
 'listings': 1,
 'issuing': 1,
 'process': 1,
 'transaction': 1,
 'dollar': 13,
 'correction': 1,
 'mark': 2,
 'u': 10,
 'chemical': 1,
 'cocoa': 1,
 'storage': 1,
 'parties': 1,
 'company': 11,
 'italian': 1,
 'danger': 1,
 'talks': 2,
 'first': 1,
 'move': 2,
 'market': 2,
 'negotiation': 1,
 'value': 1,
 'only': 2,
 'discount': 1,
 'bundesbank': 2,
 'commission': 1,
 'current': 4,
 'solidarity': 1,
 'markets': 4,
 'changes': 1,
 'key': 2,
 'average': 2,
 'last': 1,
 'ability': 1,
 'impact': 1,
 'total': 1,
 'german': 1,
 'substantial': 1,
 'reagan': 2,
 'bill': 1,
 'financial': 2,
 'finance': 1,
 'full': 1,
 'it': 1,
 'consumer': 1,
 'government': 8,
 'monthly': 1,
 'so': 1,
 'soviet': 2,
 'fire': 1,
 'securing': 1,
 'second': 2,
 'system': 2,
 'fund': 1,
 'industrial': 1,
 'fed': 19,
 'latest': 1,
 'yen': 2,
 'controlled': 1,
 'foreign': 1,
 'tariffs': 2,
 'group': 2,
 'much': 1,
 'worst': 1,